# 02a - Channel overview

Chooses which images form the analysis cube and inspects their relationships
before any modelling.

> This notebook is a thin wrapper around the `src/` package. Every computation
> below is the same function the CLI calls, so the notebook and
> `python scripts/run_pipeline.py` produce identical results. To change a
> parameter, edit `configs/pipeline_config.yaml` rather than the code here.

In [ ]:
import sys
from pathlib import Path

# Locate the project root so `src` imports work wherever Jupyter was started from.
ROOT = Path.cwd()
while not (ROOT / "configs" / "pipeline_config.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

%matplotlib inline
from src.config import load_config

config = load_config()
print("Raw data :", config.raw_dir)
print("Outputs  :", config.processed_dir)

## Choose the analysis channels

The most common pixel grid becomes the analysis grid, and the polarity that
dominates it becomes the analysis polarity. Everything else is treated as a
secondary acquisition. Both choices can be pinned in the config.

In [ ]:
from src.features.selection import build_cube, select_analysis_channels
from src.preprocessing.stack_io import load_clean_stack

images, metadata = load_clean_stack(config.processed_dir)
selection = select_analysis_channels(images, metadata, config)

print(selection.describe())
for key, label in zip(selection.keys, selection.labels):
    print(f"  {label:22s} <- {key}")

In [ ]:
cube = build_cube(images, selection)
print("analysis cube:", cube.shape)

## Channel series and composites

In [ ]:
from src.viz import overview

overview.plot_channel_series(
    images, selection, config.figure_path("02a_channel_series.png"), config
)
overview.plot_rgb_composites(
    images, selection, config.figure_path("02a_rgb_composites.png"), config
)
print("figures written")

## Inter-channel correlation

Strongly correlated channels carry redundant spatial information, which is
what makes the decomposition in 02b worthwhile.

In [ ]:
import pandas as pd

matrix = overview.channel_correlation_matrix(cube)
overview.plot_correlation_matrix(
    matrix, selection.labels, config.figure_path("02a_correlation_matrix.png")
)
pd.DataFrame(matrix, index=selection.labels, columns=selection.labels).round(3)

## Compare the acquisitions

In [ ]:
if selection.secondary_keys:
    overview.plot_polarity_comparison(
        images, selection, config.figure_path("02a_polarity_comparison.png")
    )
    overview.plot_mean_comparison(
        images, selection, config.figure_path("02a_polarity_mean_comparison.png")
    )
    print("figures written")
else:
    print("No secondary acquisition in this dataset.")

Equivalent CLI command:

```bash
python scripts/run_pipeline.py --stage overview
```